# ДЗ07 — Domain shift RGB→IR (VisDrone → HIT-UAV)

**Автор:** Serhii Polishchuk  

| Крок | Зміст |
|------|--------|
| 1 | Два мапінги (eval / train) |
| 2 | Zero-shot на HIT val |
| 3 | Інтервенції A, B, A+B |
| 4 | Зведена таблиця + small objects |
| 5 | Висновок |

Порядок: **zero → A → B → A+B**. Один evaluator (imgsz/conf/iou).

In [ ]:
!pip install -q ultralytics opencv-python-headless pyyaml

import os, json, random, shutil, hashlib, zipfile, urllib.request
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import cv2
import yaml
import torch
from PIL import Image
from tqdm.auto import tqdm
from ultralytics import YOLO

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

IMGSZ = 640
CONF = 0.25
IOU = 0.7
EPOCHS_A = 15   # grayscale VisDrone subset
EPOCHS_B = 20   # HIT subset
EPOCHS_AB = 15  # HIT from weights A
SUBSET_N = 150  # 100–200 кадрів

OUT = Path("./dz07_out")
OUT.mkdir(exist_ok=True)
(OUT / "weights").mkdir(exist_ok=True)
(OUT / "lists").mkdir(exist_ok=True)
(OUT / "viz").mkdir(exist_ok=True)

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## 1. Мапінги (фіксуємо до будь-якого запуску)

### Оцінювання: VisDrone pred → 3 класи HIT-UAV
| HIT-UAV | VisDrone class ids |
|---------|-------------------|
| Person | pedestrian(0), people(1) |
| Car | car(3) |
| Bicycle | bicycle(2) |

### Навчання: HIT category → YOLO class_id у мітках
| HIT cat | YOLO id у labels | VisDrone-аналог |
|---------|------------------|-----------------|
| Person(0) | **0** | pedestrian |
| Car(1) | **3** | car |
| Bicycle(2) | **2** | bicycle |
| OtherVehicle/DontCare | не пишемо | — |

У yaml names — як у ДЗ1 (повний COCO/VisDrone список або той самий), щоб не змінювалась голова.

In [ ]:
# --- EVAL mapping: VisDrone cls → HIT report name ---
EVAL_VD_TO_HIT = {
    0: "Person",   # pedestrian
    1: "Person",   # people
    2: "Bicycle",  # bicycle
    3: "Car",      # car
    # van, truck, bus, motor, ... → ignore for 3-class metrics
}
HIT_NAMES_3 = ["Person", "Car", "Bicycle"]
HIT_NAME_TO_IDX = {"Person": 0, "Car": 1, "Bicycle": 2}

# --- TRAIN mapping: HIT category_id → YOLO class_id in label files ---
# HIT JSON often: 0 Person, 1 Car, 2 Bicycle, 3 OtherVehicle, 4 DontCare
TRAIN_HIT_TO_YOLO = {
    0: 0,  # Person → 0 (pedestrian slot)
    1: 3,  # Car → 3
    2: 2,  # Bicycle → 2
    # 3, 4 skipped
}

# Also string names if annotations use names
TRAIN_HIT_NAME_TO_YOLO = {
    "person": 0, "Person": 0,
    "car": 3, "Car": 3,
    "bicycle": 2, "Bicycle": 2,
}

print("EVAL map:", EVAL_VD_TO_HIT)
print("TRAIN map:", TRAIN_HIT_TO_YOLO)

## 2. Дані: HIT-UAV + VisDrone

**HIT-UAV:** поклади розпакований датасет у `./datasets/HIT-UAV/`  
(з GitHub/Zenodo/Kaggle: images + annotations JSON або вже YOLO).  

Очікувані варіанти структури (ноутбук підхопить):
- `images/train`, `images/val` + `labels/train`, `labels/val` (YOLO)
- або COCO/JSON `annotations/` + `images/`

**VisDrone train** — для інтервенції A (grayscale subset).

In [ ]:
HIT_ROOT = Path("./datasets/HIT-UAV")
VD_ROOT = Path("./datasets/VisDrone")

# --- VisDrone DET val/train (як у попередніх ДЗ) ---
def ensure_visdrone_val():
    target = VD_ROOT / "VisDrone2019-DET-val"
    if target.exists() and any(target.rglob("*.jpg")):
        return target
    VD_ROOT.mkdir(parents=True, exist_ok=True)
    url = "https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-val.zip"
    zpath = VD_ROOT / "val.zip"
    print("Downloading VisDrone val...")
    urllib.request.urlretrieve(url, zpath)
    with zipfile.ZipFile(zpath) as zf:
        zf.extractall(VD_ROOT)
    zpath.unlink(missing_ok=True)
    return target

vd_val = ensure_visdrone_val()
print("VisDrone val:", vd_val)

# HIT check
print("HIT-UAV root exists:", HIT_ROOT.exists())
if HIT_ROOT.exists():
    for p in sorted(HIT_ROOT.rglob("*"))[:20]:
        if p.is_file():
            print(" ", p.relative_to(HIT_ROOT))
else:
    print("""
>>> ПОКЛАДИ HIT-UAV у ./datasets/HIT-UAV/
Джерела:
  - https://github.com/suojiashun/HIT-UAV-Infrared-Thermal-Dataset
  - Zenodo: https://zenodo.org/records/7633134
  - Kaggle: HIT-UAV YOLO format
Після розпаковки перезапусти цю клітинку.
""")

In [ ]:
def find_hit_split(root: Path):
    """Return dict train/val with image dirs and optional label dirs or json."""
    info = {}
    # YOLO layout
    for split in ["train", "val", "test"]:
        for cand in [root / "images" / split, root / split / "images", root / split]:
            if cand.exists() and any(cand.glob("*.jpg")) or any(cand.glob("*.png")):
                lab = None
                for lc in [root / "labels" / split, root / split / "labels", cand.parent / "labels"]:
                    if lc.exists():
                        lab = lc
                        break
                info[split] = {"images": cand, "labels": lab}
                break
    # flat images
    if not info:
        imgs = list(root.rglob("*.jpg")) + list(root.rglob("*.png"))
        if imgs:
            info["all"] = {"images": root, "labels": None, "files": imgs}
    return info

hit_info = find_hit_split(HIT_ROOT) if HIT_ROOT.exists() else {}
print("HIT splits found:", list(hit_info.keys()))
for k, v in hit_info.items():
    n = len(list(v["images"].glob("*.jpg"))) + len(list(v["images"].glob("*.png")))
    print(f"  {k}: {n} images, labels={v.get('labels')}")

In [ ]:
def convert_hit_json_to_yolo(json_path: Path, images_dir: Path, out_labels: Path):
    """COCO-like HIT JSON → YOLO labels with TRAIN remap (0/3/2). Skip Other/DontCare."""
    out_labels.mkdir(parents=True, exist_ok=True)
    data = json.loads(json_path.read_text())
    # build image id → file + size
    id2img = {}
    for im in data.get("images", []):
        id2img[im["id"]] = im
    # category id map from annotation categories
    cat_map = {}
    for c in data.get("categories", []):
        name = c.get("name", "")
        if name.lower() in ("person",):
            cat_map[c["id"]] = 0
        elif name.lower() in ("car",):
            cat_map[c["id"]] = 3
        elif name.lower() in ("bicycle",):
            cat_map[c["id"]] = 2
        # else skip

    by_img = defaultdict(list)
    for ann in data.get("annotations", []):
        cid = ann.get("category_id")
        yolo_cls = cat_map.get(cid)
        if yolo_cls is None:
            # try TRAIN_HIT_TO_YOLO on raw id
            yolo_cls = TRAIN_HIT_TO_YOLO.get(cid)
        if yolo_cls is None:
            continue
        im = id2img.get(ann["image_id"])
        if not im:
            continue
        W, H = im.get("width"), im.get("height")
        if not W or not H:
            continue
        # bbox xywh absolute
        x, y, w, h = ann["bbox"]
        xc, yc = (x + w / 2) / W, (y + h / 2) / H
        bw, bh = w / W, h / H
        by_img[im["file_name"]].append(f"{yolo_cls} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}\n")

    n = 0
    for fname, lines in by_img.items():
        stem = Path(fname).stem
        (out_labels / f"{stem}.txt").write_text("".join(lines))
        n += 1
    print(f"Converted {n} label files → {out_labels}")
    return out_labels

# If labels missing but JSON present — convert
if HIT_ROOT.exists():
    jsons = list(HIT_ROOT.rglob("*.json"))
    print("JSON files:", len(jsons))
    for jp in jsons[:5]:
        print(" ", jp.relative_to(HIT_ROOT))

In [ ]:
# Побудова YOLO dataset layout для HIT val/train з remap
HIT_YOLO = Path("./datasets/HIT-UAV-yolo")

def build_hit_yolo_from_existing():
    """Copy/link images and ensure labels use TRAIN remap."""
    if not hit_info:
        print("No HIT data found")
        return None
    for split in ["train", "val", "test"]:
        if split not in hit_info:
            continue
        src_img = hit_info[split]["images"]
        src_lab = hit_info[split].get("labels")
        dst_img = HIT_YOLO / "images" / split
        dst_lab = HIT_YOLO / "labels" / split
        dst_img.mkdir(parents=True, exist_ok=True)
        dst_lab.mkdir(parents=True, exist_ok=True)

        imgs = list(src_img.glob("*.jpg")) + list(src_img.glob("*.png"))
        for ip in tqdm(imgs, desc=f"HIT {split}"):
            dst = dst_img / ip.name
            if not dst.exists():
                try:
                    os.symlink(ip.resolve(), dst)
                except Exception:
                    shutil.copy(ip, dst)
            # labels
            lp = None
            if src_lab:
                for ext in [".txt"]:
                    cand = src_lab / (ip.stem + ext)
                    if cand.exists():
                        lp = cand
                        break
            out_lp = dst_lab / (ip.stem + ".txt")
            if lp and not out_lp.exists():
                # remap class ids if labels are HIT-native 0,1,2
                lines_out = []
                for line in lp.read_text().strip().splitlines():
                    p = line.split()
                    if len(p) < 5:
                        continue
                    cid = int(float(p[0]))
                    # if already YOLO 0/2/3 keep; if HIT 0/1/2 remap
                    if cid in (0, 1, 2) and cid != 3:
                        # ambiguous: assume HIT ids if max class in file <= 4
                        yid = TRAIN_HIT_TO_YOLO.get(cid, None)
                        if yid is None:
                            continue
                        cid = yid
                    if cid not in (0, 2, 3):
                        continue
                    lines_out.append(f"{cid} " + " ".join(p[1:]) + "\n")
                out_lp.write_text("".join(lines_out))
            elif not out_lp.exists():
                out_lp.write_text("")  # empty allowed
    return HIT_YOLO

if HIT_ROOT.exists() and hit_info:
    build_hit_yolo_from_existing()
    print("HIT YOLO layout:", HIT_YOLO)
else:
    print("Skip HIT YOLO build — add dataset first")

In [ ]:
# YAML з names як у VisDrone/COCO (не змінюємо голову)
# Для val HIT використовуємо images/val + labels з remap

def write_data_yaml(path, train_rel, val_rel, root):
    # names: full 80 COCO to keep head compatible with yolov8n pretrained / DZ1
    tmp = YOLO("yolov8n.pt")
    names_block = "\n".join([f"  {i}: {n}" for i, n in tmp.names.items()])
    text = f"""
path: {Path(root).resolve()}
train: {train_rel}
val: {val_rel}

names:
{names_block}
""".strip() + "\n"
    Path(path).write_text(text)
    print("Wrote", path)
    return path

HIT_YAML = OUT / "hit_uav.yaml"
if (HIT_YOLO / "images" / "val").exists():
    write_data_yaml(HIT_YAML, "images/train", "images/val", HIT_YOLO)
elif (HIT_YOLO / "images" / "test").exists():
    write_data_yaml(HIT_YAML, "images/train", "images/test", HIT_YOLO)
else:
    print("HIT yaml pending — dataset not ready")

## 3. Zero-shot (RGB→IR)

Ваги: checkpoint ДЗ1 (`best.pt`) або `yolov8n.pt` якщо немає.

Метрики рахуємо через `val`, потім **фільтруємо класи** до Person/Car/Bicycle за EVAL map.

In [ ]:
WEIGHTS_DZ1 = "yolov8n.pt"
for c in [Path("/content/best.pt"), Path("./best.pt"),
          Path("../DZ03_Vision_Serhii Polishchuk/best.pt"),
          Path("./dz07_out/weights/best_dz1.pt")]:
    if c.exists():
        WEIGHTS_DZ1 = str(c)
        break
print("Zero-shot weights:", WEIGHTS_DZ1)

if not HIT_YAML.exists():
    print("STOP: спочатку підготуй HIT dataset і hit_uav.yaml")
else:
    model0 = YOLO(WEIGHTS_DZ1)
    res0 = model0.val(data=str(HIT_YAML), imgsz=IMGSZ, conf=CONF, iou=IOU,
                      split="val", plots=False, verbose=True)
    # overall box metrics (includes all classes model predicts)
    zero_metrics = {
        "run": "zero",
        "mAP50": float(res0.box.map50),
        "mAP50-95": float(res0.box.map),
        "precision": float(res0.box.mp),
        "recall": float(res0.box.mr),
    }
    # per-class if available
    names = model0.names
    if hasattr(res0.box, "ap_class_index") and res0.box.ap_class_index is not None:
        for i, ci in enumerate(res0.box.ap_class_index):
            cname = names.get(int(ci), str(ci))
            zero_metrics[f"AP50_{cname}"] = float(res0.box.ap50[i]) if hasattr(res0.box, "ap50") else None
    print("ZERO:", zero_metrics)
    pd.DataFrame([zero_metrics]).to_csv(OUT / "metrics_zero.csv", index=False)

In [ ]:
# 2–3 кадри viz: GT + pred
def viz_predictions(model, img_paths, out_dir, tag, n=3):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    paths = list(img_paths)[:n]
    for ip in paths:
        r = model.predict(str(ip), imgsz=IMGSZ, conf=CONF, iou=IOU, verbose=False)[0]
        plotted = r.plot()  # BGR numpy
        cv2.imwrite(str(out_dir / f"{tag}_{Path(ip).stem}.jpg"), plotted)
        print("Saved", out_dir / f"{tag}_{Path(ip).stem}.jpg")

if HIT_YAML.exists() and (HIT_YOLO / "images" / "val").exists():
    val_imgs = sorted((HIT_YOLO / "images" / "val").glob("*.jpg"))[:50]
    if not val_imgs:
        val_imgs = sorted((HIT_YOLO / "images" / "val").glob("*.png"))[:50]
    viz_predictions(model0, val_imgs, OUT / "viz", "zero", n=3)

## 4. Інтервенція A — grayscale VisDrone subset

Гіпотеза: зсув частково через відсутність кольору.

In [ ]:
# Потрібен VisDrone train. Якщо немає — можна взяти val як proxy лише для відладки (НЕ для здачі).
VD_TRAIN = VD_ROOT / "VisDrone2019-DET-train"
GRAY = Path("./datasets/VisDrone-gray-subset")

def build_gray_subset(src_images, src_labels, n=SUBSET_N, seed=SEED):
    GRAY.mkdir(parents=True, exist_ok=True)
    (GRAY / "images" / "train").mkdir(parents=True, exist_ok=True)
    (GRAY / "labels" / "train").mkdir(parents=True, exist_ok=True)
    imgs = sorted(list(Path(src_images).glob("*.jpg")))
    rng = random.Random(seed)
    pick = rng.sample(imgs, min(n, len(imgs)))
    list_path = OUT / "lists" / "subset_A_visdrone_gray.txt"
    lines = []
    for ip in tqdm(pick, desc="gray A"):
        img = cv2.imread(str(ip), cv2.IMREAD_GRAYSCALE)
        img3 = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)  # 3 identical channels
        dst = GRAY / "images" / "train" / ip.name
        cv2.imwrite(str(dst), img3)
        lp = Path(src_labels) / (ip.stem + ".txt")
        out_l = GRAY / "labels" / "train" / (ip.stem + ".txt")
        if lp.exists():
            shutil.copy(lp, out_l)
        else:
            out_l.write_text("")
        lines.append(str(dst.resolve()) + "\n")
    list_path.write_text("".join(lines))
    print("A list:", list_path, "n=", len(pick))
    return GRAY

# labels_coco from previous HW if present
vd_img_src = vd_val / "images"
vd_lab_src = vd_val / "labels_coco" if (vd_val / "labels_coco").exists() else vd_val / "labels"
if VD_TRAIN.exists():
    vd_img_src = VD_TRAIN / "images"
    vd_lab_src = VD_TRAIN / "labels"

print("A source images:", vd_img_src)
if vd_img_src.exists():
    build_gray_subset(vd_img_src, vd_lab_src, n=SUBSET_N)
    # val for A training still HIT val
    GRAY_YAML = OUT / "visdrone_gray.yaml"
    write_data_yaml(GRAY_YAML, str((GRAY / "images" / "train").resolve()),
                    str((HIT_YOLO / "images" / "val").resolve()) if (HIT_YOLO / "images" / "val").exists()
                    else str((GRAY / "images" / "train").resolve()),
                    ".")
    # fix yaml paths manually simpler:
    tmp = YOLO("yolov8n.pt")
    names_block = "\n".join([f"  {i}: {n}" for i, n in tmp.names.items()])
    GRAY_YAML.write_text(f"""
path: {(GRAY).resolve()}
train: images/train
val: images/train

names:
{names_block}
""".strip()+"\n")
    print(GRAY_YAML.read_text()[:300])

In [ ]:
# Train A
if (GRAY / "images" / "train").exists():
    model_a = YOLO(WEIGHTS_DZ1)
    # For eval on HIT we need separate val yaml — train on gray, later val on HIT
    train_a = model_a.train(
        data=str(GRAY_YAML),
        imgsz=IMGSZ,
        epochs=EPOCHS_A,
        batch=16,
        seed=SEED,
        project=str(OUT / "runs"),
        name="A_gray_visdrone",
        exist_ok=True,
        verbose=True,
    )
    best_a = Path(OUT / "runs" / "A_gray_visdrone" / "weights" / "best.pt")
    if best_a.exists():
        shutil.copy(best_a, OUT / "weights" / "A_best.pt")
        h = hashlib.sha256(best_a.read_bytes()).hexdigest()[:16]
        (OUT / "weights" / "A_best.sha256").write_text(h + "\n")
        print("A best hash:", h)
else:
    print("Skip A train — no gray subset")

## 5. Інтервенція B — HIT train subset (з ваг ДЗ1)

Гіпотеза: мала in-domain вибірка закриває зсув.

In [ ]:
def make_hit_subset_list(split="train", n=SUBSET_N, seed=SEED):
    img_dir = HIT_YOLO / "images" / split
    if not img_dir.exists():
        # try test as train proxy only if needed
        print("No", img_dir)
        return None
    imgs = sorted(list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.png")))
    rng = random.Random(seed)
    pick = rng.sample(imgs, min(n, len(imgs)))
    # write subset folders
    sub = Path(f"./datasets/HIT-subset-{split}")
    (sub / "images" / "train").mkdir(parents=True, exist_ok=True)
    (sub / "labels" / "train").mkdir(parents=True, exist_ok=True)
    list_path = OUT / "lists" / f"subset_B_hit_{split}.txt"
    lines = []
    for ip in pick:
        dst = sub / "images" / "train" / ip.name
        if not dst.exists():
            try:
                os.symlink(ip.resolve(), dst)
            except Exception:
                shutil.copy(ip, dst)
        lp = HIT_YOLO / "labels" / split / (ip.stem + ".txt")
        out_l = sub / "labels" / "train" / (ip.stem + ".txt")
        if lp.exists():
            shutil.copy(lp, out_l)
        else:
            out_l.write_text("")
        lines.append(str(dst.resolve()) + "\n")
    list_path.write_text("".join(lines))
    print("B list:", list_path, "n=", len(pick))
    return sub, list_path

if (HIT_YOLO / "images" / "train").exists():
    hit_sub, hit_list = make_hit_subset_list("train", n=SUBSET_N)
elif (HIT_YOLO / "images" / "val").exists():
    print("WARNING: using val images for B subset only if train missing — better add train")
    hit_sub, hit_list = make_hit_subset_list("val", n=SUBSET_N)
else:
    hit_sub = None
    print("No HIT images for B")

In [ ]:
if hit_sub is not None:
    tmp = YOLO("yolov8n.pt")
    names_block = "\n".join([f"  {i}: {n}" for i, n in tmp.names.items()])
    HIT_SUB_YAML = OUT / "hit_subset_B.yaml"
    # train on subset; val on full HIT val if available
    val_rel = "images/train"
    HIT_SUB_YAML.write_text(f"""
path: {hit_sub.resolve()}
train: images/train
val: images/train

names:
{names_block}
""".strip()+"\n")

    model_b = YOLO(WEIGHTS_DZ1)  # same start as zero, NOT from A
    train_b = model_b.train(
        data=str(HIT_SUB_YAML),
        imgsz=IMGSZ,
        epochs=EPOCHS_B,
        batch=16,
        seed=SEED,
        project=str(OUT / "runs"),
        name="B_hit",
        exist_ok=True,
        verbose=True,
    )
    best_b = Path(OUT / "runs" / "B_hit" / "weights" / "best.pt")
    if best_b.exists():
        shutil.copy(best_b, OUT / "weights" / "B_best.pt")
        h = hashlib.sha256(best_b.read_bytes()).hexdigest()[:16]
        (OUT / "weights" / "B_best.sha256").write_text(h + "\n")
        print("B best hash:", h)

## 6. Інтервенція A+B — HIT subset з ваг після A

In [ ]:
weights_a = OUT / "weights" / "A_best.pt"
if weights_a.exists() and hit_sub is not None:
    model_ab = YOLO(str(weights_a))
    train_ab = model_ab.train(
        data=str(HIT_SUB_YAML),  # same file list as B
        imgsz=IMGSZ,
        epochs=EPOCHS_AB,
        batch=16,
        seed=SEED,
        project=str(OUT / "runs"),
        name="AB_hit_from_A",
        exist_ok=True,
        verbose=True,
    )
    best_ab = Path(OUT / "runs" / "AB_hit_from_A" / "weights" / "best.pt")
    if best_ab.exists():
        shutil.copy(best_ab, OUT / "weights" / "AB_best.pt")
        h = hashlib.sha256(best_ab.read_bytes()).hexdigest()[:16]
        (OUT / "weights" / "AB_best.sha256").write_text(h + "\n")
        print("A+B best hash:", h)
else:
    print("Skip A+B — need A_best.pt and HIT subset")

## 7. Повторне оцінювання zero / A / B / A+B на HIT val

In [ ]:
def eval_weights(tag, weights_path):
    if not Path(weights_path).exists():
        print("Missing", weights_path)
        return None
    if not HIT_YAML.exists():
        print("No HIT_YAML")
        return None
    m = YOLO(str(weights_path))
    r = m.val(data=str(HIT_YAML), imgsz=IMGSZ, conf=CONF, iou=IOU,
              split="val", plots=False, verbose=False)
    row = {
        "run": tag,
        "mAP50": float(r.box.map50),
        "mAP50-95": float(r.box.map),
        "precision": float(r.box.mp),
        "recall": float(r.box.mr),
    }
    print(tag, row)
    return row

rows = []
# zero already may exist
if (OUT / "metrics_zero.csv").exists():
    rows.append(pd.read_csv(OUT / "metrics_zero.csv").iloc[0].to_dict())
else:
    z = eval_weights("zero", WEIGHTS_DZ1)
    if z: rows.append(z)

for tag, wp in [("A", OUT / "weights" / "A_best.pt"),
                ("B", OUT / "weights" / "B_best.pt"),
                ("A+B", OUT / "weights" / "AB_best.pt")]:
    row = eval_weights(tag, wp)
    if row:
        rows.append(row)

df = pd.DataFrame(rows)
display(df)
df.to_csv(OUT / "metrics_all.csv", index=False)
print("Saved", OUT / "metrics_all.csv")

### Висновок (шаблон 6–10 речень)

1. Найбільший зсув zero-shot між класами: … (Car / Person / Bicycle).  
2. A (grayscale): чи підтримала гіпотезу «лише колір»?  
3. B (HIT subset): наскільки закриває зсув vs zero.  
4. A+B vs B: стабільний виграш чи шум?  
5. Які ваги лишаємо для теми 6 і чому.  
6. Вплив на дрібні об’єкти (Person ~10–15 px).  
7. Обмеження: розмір підмножини, епохи, версія HIT-UAV.